In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import json
import sys

import numpy as np
# Resolve project/backend roots from notebook location.
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd.parent, _cwd.parent.parent, _cwd.parent.parent.parent]
project_root = None
for p in _candidates:
    if (p / "backend").exists():
        project_root = p
        break
if project_root is None:
    raise RuntimeError("Could not resolve project root containing backend/.")

backend_root = project_root / "backend"
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

from autonomous_control.config.randomness import RandomnessConfig, apply_global_seed, derive_seed
from autonomous_control.controller_agent import MPOAgent
from autonomous_control.episode_runner import EpisodeRunner
from autonomous_control.mpo_config import MPOConfig
from autonomous_control.training_runtime import make_attitude_control_env
from environment_definition.constants import RenderMode
from environment_definition.mission_profiles.s01_multiple_targets_fwd_fish import build_setup, sample_satellite_altitude
from render.render_main import render_from_series
from utils.ml_training.ml_training_utils import create_run_dir, init_run_markdown, append_run_markdown_event

#get the helper functions to display videos and capture logs
from utils.notebook.video import play_saved_video, init_video_cell, export_and_play_saved_video, _export_render_video
from utils.notebook.episode_log import _episode_metrics, _write_csv, EpisodeArtifact, _warmup_peak_index



SEED = 7
VIDEOS_PER_CELL = 1
RNG_CFG = RandomnessConfig(seed=SEED)
apply_global_seed(RNG_CFG)
SATELLITE_ALTITUDE = sample_satellite_altitude(seed=derive_seed(SEED, "mission_altitude"))

RUN_ID = f"nb-mpo-{datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S')}"
RUN_DIR = create_run_dir(run_id=RUN_ID)

init_run_markdown(
    RUN_DIR,
    title="Notebook MPO Workflow",
    metadata={
        "seed": SEED,
        "sampled_altitude_km": float(SATELLITE_ALTITUDE.to("km").magnitude),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    },
)

print(f"RUN_DIR: {RUN_DIR}")
print(f"Sampled altitude: {SATELLITE_ALTITUDE}")

warmup_episode_count = 1
train_episode_count = 1
test_episode_count = 1

print(f"""
##########################################
##########################################
Warmup episodes: {warmup_episode_count}
#################################
Training ep:     {train_episode_count}
#################################
Test ep:         {test_episode_count}
##########################################
##########################################
""")


RUN_DIR: D:\code\sem-proj-asc\backend\autonomous_control\models\nb-mpo-2026-05-19_10-42-33
Sampled altitude: 528.758 km

##########################################
##########################################
Warmup episodes: 1
#################################
Training ep:     1
#################################
Test ep:         1
##########################################
##########################################



these new constraints need to be wired in somehow:

# New Constraints

## sat model
### control / safety
- max 30 deg sweep in all directions -> encode as RW safety mechanism:
- create safe mode to comply with safety restrictions
#### safety torque cmd constraints
- - if in (30,35) interval reduce torque gradually
#### safe mode activation
- -  if over 35 deg abort and go into "safe mode" -> returning to nadir
- - if over 30 deg and safe mode activation at 35 would let the sat exceed 45 deg due to momentum
- - stay in nadir for 10 s (no external torque control mode allowed)

### 2nd vision sensor
- make an educated guess of what fish eye placement would be useful -> add to 2nd camera init
- educated guess of what FOV for the fish eye would be useful -> add to 2nd camera init

## environment:
- clouds cover a larger area
- clouds move
- vary in height

## mission:
- max 10 images per orbit (memory, downlink constraint)
- capture as many high value observations over a cloud covered area as possible -> benchmark
- target grid, 50 cells (many areas)
- - decide on size and location
- images with clouds get less / no reward as not useful
- images need to be captured with fwd motion compensation, get more reward if not blurry

- - implement image capturing mode (when control decides to do the image capturing on a target )

## RL / reward mechanism / KPI:
- calulate a benchmark using a "dummy strategy" for the env initiation

## control agent
- give 2nd camera array
- add action space: "capture image" -> records the "camera ground speed" over fixed time interval starting from action input ending after 2s,



In [38]:
# Secondary forward-looking camera (GoPro 4K+ mental model) — s01 default: 25° prograde tilt
# Tilt = 25° forward along-track (§A): positive tilt = prograde, boresight shifted ahead of nadir.
# build_setup() sets this automatically; this cell just displays the optics.
from environment_definition.constants.SATELLITE import CAMERA_ALTITUDE
from environment_definition.constants.UNIT_REGISTRY import UREG as ureg
from simulation.camera_image import CameraImage, CameraMount

TILT_ANGLE_SCND = 25 * ureg.deg  # s01 default: 25° forward-looking (§A)
FOV_ANGLE_SCND = 60 * ureg.deg
N_PIXELS_SCND_X = 5312
N_PIXELS_SCND_Y = 2988
PIXEL_SIZE_SCND = 1.55 * ureg.um  # educated guess; TBD from datasheet

scnd_camera = CameraImage.from_fov(
    fov_y=FOV_ANGLE_SCND,
    pixel_size=PIXEL_SIZE_SCND,
    n_pixels_x=N_PIXELS_SCND_X,
    n_pixels_y=N_PIXELS_SCND_Y,
    axis="x"
)
scnd_mount = CameraMount(camera=scnd_camera, tilt_off_nadir=TILT_ANGLE_SCND)

ref_altitude = SATELLITE_ALTITUDE if "SATELLITE_ALTITUDE" in globals() else CAMERA_ALTITUDE
gsd = scnd_camera.gsd_at(ref_altitude)
swath_y = scnd_camera.swath_at(ref_altitude, axis="y")
swath_x = scnd_camera.swath_at(ref_altitude, axis="x")

print("Secondary camera (GoPro-style)")
print(f"  mount tilt off nadir: {scnd_mount.tilt_off_nadir.to('deg'):~}")
print(f"  focal length:         {scnd_camera.focal_length.to('mm'):~}")
print(f"  pixel pitch:          {scnd_camera.pixel_size.to('um'):~}")
print(f"  resolution:           {scnd_camera.n_pixels_x} x {scnd_camera.n_pixels_y}")
print(f"  FOV (y / x):          {scnd_camera.fov(axis='y').to('deg'):~} / {scnd_camera.fov(axis='x').to('deg'):~}")
print(f"  GSD @ {ref_altitude.to('km'):~}:           {gsd.to('m'):~}")
print(f"  swath (y / x):        {swath_y.to('km'):~} / {swath_x.to('km'):~}")


Secondary camera (GoPro-style)
  mount tilt off nadir: 25 deg
  focal length:         7.130506764599555 mm
  pixel pitch:          1.55 µm
  resolution:           5312 x 2988
  FOV (y / x):          35.98339777135763 deg / 59.99999999999999 deg
  GSD @ 528.7583065070219 km:           114.93928862879507 m
  swath (y / x):        343.4385944228396 km / 610.5575011961594 km


In [ ]:
# Step 1 – single warmup smoke test
saved_videos: list[Path] = []

setup = build_setup(seed=derive_seed(SEED, "mission_altitude"))
_resolved_for_env = setup.resolve(require_camera=False)
warmup_env = make_attitude_control_env(
    secondary_camera_observation_line_n_bins=_resolved_for_env.secondary_camera_observation_line_n_bins
)
warmup_agent = MPOAgent(warmup_env, config=MPOConfig(warmup_episodes=0))

ep_rng = np.random.default_rng(derive_seed(SEED, "nb_warmup", 0))
result = EpisodeRunner(setup).run_serial(
    warmup_agent,
    mode="warmup",
    warmup_controller="random",
    np_rng=ep_rng,
)

total, avg, steps = _episode_metrics(result)
print(f"warmup: steps={steps} total_reward={total:.3f} avg_reward={avg:.3f}")

video_path = RUN_DIR / "warmup_01.mp4"
_export_render_video(simulation_series=result.simulation_series, out_path=video_path)
saved_videos.append(video_path)
print(f"Saved: {video_path}")




Using device: cuda
[run_serial] start mode=warmup max_steps=1883             
[run_serial] end mode=warmup steps=1883 total_reward=-177840.312928 avg_reward=-94.445201            
warmup: steps=1883 total_reward=-177840.313 avg_reward=-94.445
[mpo_video:after_export] warmup_01.mp4 (11148250 bytes)
Saved: D:\code\sem-proj-asc\backend\autonomous_control\models\nb-mpo-2026-05-19_10-42-33\warmup_01.mp4


In [4]:
# Play saved videos from disk (no re-export)
for video_path in saved_videos:
    print(f"Loading from:\n {video_path}")
    play_saved_video(video_path)


Loading from:
 D:\code\sem-proj-asc\backend\autonomous_control\models\nb-mpo-2026-05-19_09-40-09\warmup_01.mp4
[mpo_video:play] warmup_01.mp4 (11148250 bytes)


In [5]:

# ── S01 Coast Benchmark ────────────────────────────────────────────────────────
# Two independent benchmark knobs (both active here):
#   1. Nadir init: build_setup() sets sat_z_offset=0° (body +Z at Earth center on episode start)
#   2. Coast controller: controller_mode="coast" → always 0 N·m reaction-wheel torque
#
# This gives a deterministic reference episode for comparing RL/warmup performance.
# Use coast for benchmark; switch to controller_mode="random" if you want dramatic clouds for video.

from environment_definition.constants.SIMULATION import RenderMode, SimulationConfig
from environment_definition.mission_profiles.s01_multiple_targets_fwd_fish import build_setup
from simulation.stepper_factory import build_stepper
from simulation.stepper import run_baseline_rollout_from_stepper
from environment_definition.constants.UNIT_REGISTRY import UREG as ureg

BENCH_SEED = 7

setup = build_setup(seed=BENCH_SEED, include_cameras=True)
resolved = setup.resolve(require_camera=True)

print(f"Altitude:          {resolved.altitude.to('km'):.2f~}")
print(f"Clouds:            {len(resolved.clouds)} cloud(s)")
print(f"Cameras:           {len(resolved.cameras)} mount(s)")
print(f"Secondary bins:    {resolved.secondary_camera_observation_line_n_bins}")
print(f"sat_z_offset:      {resolved.sat_z_offset_deg:.1f}° (0° = nadir init)")

sim_cfg = SimulationConfig(render_mode=RenderMode.HEADLESS, controller_mode="coast")
stepper = build_stepper(resolved, simulation_config=sim_cfg, require_camera=True)
tau_max_nm = float(resolved.satellite.reaction_wheel_max_torque.to(ureg.N * ureg.m).magnitude)

coast_series = run_baseline_rollout_from_stepper(
    stepper, simulation_config=sim_cfg, tau_max_nm=tau_max_nm, show_progress=True
)

import numpy as np
print(f"\nCoast episode summary:")
print(f"  Steps:           {len(coast_series.t_s) - 1}")
print(f"  Mean reward:     {np.mean(coast_series.simulation_reward):.3f}")
print(f"  Primary cloud blocked (mean):    {np.nanmean(coast_series.camera_cloud_blocked_fraction):.3f}")
print(f"  Secondary cloud blocked (mean):  {np.mean(coast_series.secondary_camera_cloud_blocked_fraction):.3f}")
print(f"  Secondary obs shape:             {coast_series.secondary_camera_observation_line_codes.shape}")
print(f"  k=0 center-ray code: {int(coast_series.camera_center_ray_observation_code[0])} (1=Earth, 3=target)")
print(f"  All torques zero: {np.all(coast_series.wheel_torque_cmd_nm == 0.0)}")


Altitude:          547.51 km
Clouds:            2 cloud(s)
Cameras:           2 mount(s)
Secondary bins:    200
sat_z_offset:      0.0° (0° = nadir init)


Running simulation: 100%|██████████| 1921/1921 [00:50<00:00, 38.14step/s]


Coast episode summary:
  Steps:           1921
  Mean reward:     -92.768
  Primary cloud blocked (mean):    0.012
  Secondary cloud blocked (mean):  0.010
  Secondary obs shape:             (1922, 200)
  k=0 center-ray code: 1 (1=Earth, 3=target)
  All torques zero: True


In [25]:

# ── Export Demo Video: clouds + dual 1D strips ─────────────────────────────────
# Renders the coast episode: shows enlarged S01_CLOUDS + two 1D observation strips
# (nadir primary strip + forward-looking secondary strip, labeled "Forward (along-track)").

video_path = RUN_DIR / "coast_benchmark_dual_strip.mp4"
export_and_play_saved_video(simulation_series=coast_series, out_path=video_path)
print(f"Saved: {video_path}")


[mpo_video:after_export] coast_benchmark_dual_strip.mp4 (10870491 bytes)
Saved: D:\code\sem-proj-asc\backend\autonomous_control\models\nb-mpo-2026-05-19_10-42-33\coast_benchmark_dual_strip.mp4
[mpo_video:play] coast_benchmark_dual_strip.mp4 (10870491 bytes)


ObservationTargetArea(lat_min=<Quantity(85, 'degree')>, lat_max=<Quantity(85.0895371, 'degree')>, label='target_0')
ObservationTargetArea(lat_min=<Quantity(85.1790739, 'degree')>, lat_max=<Quantity(85.2686106, 'degree')>, label='target_1')
ObservationTargetArea(lat_min=<Quantity(85.3581469, 'degree')>, lat_max=<Quantity(85.4476831, 'degree')>, label='target_2')
ObservationTargetArea(lat_min=<Quantity(85.537219, 'degree')>, lat_max=<Quantity(85.6267547, 'degree')>, label='target_3')
ObservationTargetArea(lat_min=<Quantity(85.7162902, 'degree')>, lat_max=<Quantity(85.8058255, 'degree')>, label='target_4')
ObservationTargetArea(lat_min=<Quantity(85.8953606, 'degree')>, lat_max=<Quantity(85.9848955, 'degree')>, label='target_5')
ObservationTargetArea(lat_min=<Quantity(86.0744302, 'degree')>, lat_max=<Quantity(86.1639646, 'degree')>, label='target_6')
ObservationTargetArea(lat_min=<Quantity(86.2534989, 'degree')>, lat_max=<Quantity(86.3430331, 'degree')>, label='target_7')
ObservationTarget